![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [8]:
# QuantBook Analysis Tool 
# For more information see [https://www.quantconnect.com/docs/v2/our-platform/research/getting-started]
qb = QuantBook()
sy = qb.add_crypto("BTCUSD", Resolution.MINUTE)
# Locally Lean installs free sample data, to download more data please visit https://www.quantconnect.com/docs/v2/lean-cli/datasets/downloading-data 
qb.set_start_date(2025, 1, 1)

In [9]:
sy.symbol

In [13]:
from datetime import datetime

In [22]:
df = qb.history(sy.symbol, datetime(2017, 1, 1), datetime(2026, 1, 5))

In [23]:
len(df)

3855180

In [13]:
df = df.reset_index()
df

,symbol,time,askclose,askhigh,asklow,askopen,asksize,bidclose,bidhigh,bidlow,bidopen,bidsize,close,high,low,open,volume
0,BTCUSD,2017-09-03 00:01:00,4649.95,4649.95,4649.95,4649.95,0.400000,4642.42,4642.42,4642.42,4642.42,0.140000,4652.69,4652.71,4642.39,4642.39,6.634878
1,BTCUSD,2017-09-03 00:02:00,4652.70,4652.70,4652.70,4652.70,0.010000,4645.10,4645.10,4645.10,4645.10,0.013237,4654.33,4654.99,4645.28,4652.68,1.361324
2,BTCUSD,2017-09-03 00:03:00,4654.18,4654.18,4654.18,4654.18,0.190000,4647.30,4647.30,4647.30,4647.30,0.013236,4654.99,4654.99,4647.13,4654.16,9.681455
3,BTCUSD,2017-09-03 00:04:00,4654.99,4654.99,4654.99,4654.99,9.342381,4654.98,4654.98,4654.98,4654.98,1.819045,4659.99,4659.99,4654.99,4654.99,42.740877
4,BTCUSD,2017-09-03 00:05:00,4659.99,4659.99,4659.99,4659.99,3.393134,4659.98,4659.98,4659.98,4659.98,0.796597,4667.95,4667.95,4659.98,4659.99,16.485551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3855175,BTCUSD,2025-01-01 04:56:00,6782.61,6782.61,6782.61,6782.61,70.152113,6782.60,6782.60,6782.60,6782.60,2.261653,6619.01,6621.20,6619.01,6621.01,0.000000
3855176,BTCUSD,2025-01-01 04:57:00,6782.61,6782.61,6782.61,6782.61,70.152113,6782.60,6782.60,6782.60,6782.60,2.261653,6619.01,6621.20,6619.01,6621.01,0.000000
3855177,BTCUSD,2025-01-01 04:58:00,6782.61,6782.61,6782.61,6782.61,70.152113,6782.60,6782.60,6782.60,6782.60,2.261653,6619.01,6621.20,6619.01,6621.01,0.000000
3855178,BTCUSD,2025-01-01 04:59:00,6782.61,6782.61,6782.61,6782.61,70.152113,6782.60,6782.60,6782.60,6782.60,2.261653,6619.01,6621.20,6619.01,6621.01,0.000000


In [14]:
import pandas as pd

def analyze_time_gaps(df, time_col='time', expected_freq='1min'):
    """
    Analyzes a DataFrame for missing time gaps based on an expected frequency.
    """
    # 1. Ensure the column is datetime type and sorted chronologically
    df_sorted = df.copy()
    df_sorted[time_col] = pd.to_datetime(df_sorted[time_col])
    df_sorted = df_sorted.sort_values(by=time_col).reset_index(drop=True)
    
    # 2. Calculate the difference between consecutive timestamps
    time_diffs = df_sorted[time_col].diff()
    
    # 3. Identify where the difference is strictly greater than the expected 1 minute
    expected_delta = pd.Timedelta(expected_freq)
    gap_mask = time_diffs > expected_delta
    
    # If no gaps are found, return an empty DataFrame
    if not gap_mask.any():
        print("No gaps found!")
        return pd.DataFrame()
    
    # 4. Extract gap details
    # The gap "ends" at the row where the condition is True
    gap_ends = df_sorted.loc[gap_mask, time_col].reset_index(drop=True)
    
    # The gap "starts" at the row immediately before it
    gap_start_indices = df_sorted[gap_mask].index - 1
    gap_starts = df_sorted.loc[gap_start_indices, time_col].reset_index(drop=True)
    
    # The duration is the calculated time difference
    durations = time_diffs[gap_mask].reset_index(drop=True)
    
    # 5. Compile into a neat summary DataFrame
    gaps_summary = pd.DataFrame({
        'gap_start_time': gap_starts,
        'gap_end_time': gap_ends,
        'missing_duration': durations,
        'missing_minutes': (durations.dt.total_seconds() / 60) - 1 # Subtract 1 because a 2-min diff = 1 missing minute
    })
    
    print(f"Total gaps found: {len(gaps_summary)}")
    return gaps_summary

# ==========================================
# Example Usage:
# ==========================================

# Run the analysis
gaps_df = analyze_time_gaps(df, time_col='time')
print(gaps_df)

No gaps found!
Empty DataFrame
Columns: []
Index: []


In [20]:
len(df) / 60 / 24

2677.2083333333335

In [19]:
datetime(2025, 1, 5) - datetime(2017, 1, 1)  

datetime.timedelta(days=2926)

In [21]:
df.time.min(), df.time.max()

(Timestamp('2017-09-03 00:01:00'), Timestamp('2025-01-01 05:00:00'))